# Node-Classification Baselines - Targeted Tuning Run

Thirteen cells, all tuned (none of these five publish settings for ACM/DBLP/IMDB):

| model | datasets |
|---|---|
| CSNN | ACM, DBLP, IMDB |
| NSD | ACM, DBLP, IMDB |
| HINormer | ACM |
| H2G-former | ACM, DBLP, IMDB |
| LatGRL | ACM, DBLP, IMDB |

Protocol is HGB's transductive 20/20/60 split over five seeds, selected and reported
on Macro-F1 with Micro-F1 alongside. Every model sees the identical split: the
vendors disagree on how to carve out validation (HGB 20% random, PSHGCN 20% random,
HAN 30% sequential) and all of them are overridden centrally.

This drives `scripts/run_baseline_tuning.py`; nothing is reimplemented here. Cells
run one model at a time so a failure costs one model, not the batch.

## 0. Environment

In [1]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/H2S-NN_test


Install in two stages so a library problem cannot take the whole campaign down.

**Core** (`requirements/core.txt`) is torch, PyG and what `src/` itself needs — mandatory, and it must succeed. On a fresh Colab runtime it replaces Colab's own torch, so the cell kills the runtime afterwards; **rerun from the first cell once it restarts**. On a machine that already has the pinned versions it is a no-op and execution continues.

**Optional groups** (`requirements/baselines-*.txt`) each back a subset of the baselines. They install one pip call at a time, and a group that fails only disables the models it backs — those cells are then dropped from the campaign before anything launches, never mid-run.

In [2]:
import importlib.metadata
import os
import signal
import subprocess
import sys

# Core only. The baseline-specific groups install in the next cell, independently.
CORE_REQUIREMENTS = PROJECT_DIR / 'requirements' / 'core.txt'
if not CORE_REQUIREMENTS.exists():
    raise FileNotFoundError(f'Missing pinned requirements: {CORE_REQUIREMENTS}')

try:
    installed_torch = importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError:
    installed_torch = ''

# core.txt carries its own --extra-index-url and -f lines, so this single command pulls
# the CUDA 12.4 torch build and the matching torch_scatter/torch_sparse wheels.
target_ready = installed_torch.startswith('2.6.') and '+cu124' in installed_torch
print('installed torch distribution:', installed_torch or 'not installed')

core = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(CORE_REQUIREMENTS)],
    capture_output=True,
    text=True,
)
if core.returncode != 0:
    print(core.stdout[-3000:])
    print(core.stderr[-3000:])
    raise RuntimeError(f'Core install failed with exit code {core.returncode}.')
print('Core stack installed.')

if not target_ready:
    # Colab ships its own torch. Replacing it mid-session leaves the previously imported
    # compiled extensions loaded, so the runtime has to restart before anything else runs.
    print('Pinned Torch/CUDA stack installed. Restarting the runtime; rerun from the first cell.')
    if IN_COLAB:
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError('Restart Python, then rerun the notebook from the first cell.')

installed torch distribution: 2.6.0+cu124
Core stack installed.


In [3]:
import torch
import torch_geometric
import torch_scatter
import torch_sparse

if not torch.__version__.startswith('2.6.'):
    raise RuntimeError(f'Expected torch 2.6.*, found {torch.__version__}. Rerun the install cell.')
if torch.version.cuda != '12.4':
    raise RuntimeError(f'Expected the CUDA 12.4 Torch build, found CUDA {torch.version.cuda}.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('device:', DEVICE)
if DEVICE == 'cuda':
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name, '|', round(properties.total_memory / 1024**3, 1), 'GiB')


torch: 2.6.0+cu124
torch_geometric: 2.8.0.post1
device: cuda
GPU: NVIDIA A100-SXM4-80GB | 79.3 GiB


In [4]:
# One pip call per optional group. A failure here is reported, not raised: the models it
# backs are dropped from the campaign further down, and everything else still runs.
import subprocess
import sys

from src.baselines.runtime.dependencies import REQUIREMENT_GROUPS, refresh


def install_group(group) -> tuple[bool, str, list[str]]:
    """Install one group, retrying without build isolation if that is the failure."""
    base = [sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(group.path)]
    result = subprocess.run(base, capture_output=True, text=True)
    refresh()
    if group.is_installed():
        return True, 'installed', []
    # torch_householder ships only an sdist whose setup.py does
    # `from torch.utils.cpp_extension import ...`; pip's default build isolation hides the
    # already-installed torch from it. Retrying without isolation is the documented fix.
    retry = subprocess.run(base + ['--no-build-isolation'], capture_output=True, text=True)
    refresh()
    if group.is_installed():
        return True, 'installed (--no-build-isolation)', []
    stderr = retry.stderr or result.stderr
    tail = [line for line in stderr.splitlines() if line.strip()][-3:]
    return False, 'failed', tail


GROUP_STATUS: dict[str, bool] = {}
for group_name, requirement_group in REQUIREMENT_GROUPS.items():
    if requirement_group.is_installed():
        GROUP_STATUS[group_name] = True
        print(f'OK    {group_name:<11} already present')
        continue
    installed, detail, error_tail = install_group(requirement_group)
    GROUP_STATUS[group_name] = installed
    if installed:
        print(f'OK    {group_name:<11} {detail} - {requirement_group.summary}')
        continue
    print(f'FAIL  {group_name:<11} {requirement_group.summary}')
    print(f'      still missing: {", ".join(requirement_group.missing_modules())}')
    for line in error_tail:
        print(f'      {line[:160]}')

OK    dgl         already present
OK    sheaf       installed - Householder orthogonal maps used by the two sheaf baselines
OK    h2gb        installed - the H2GB graphgym stack behind H2G-former
OK    hetero2net  installed - Hetero^2Net's metapath-dropping and logging dependencies


In [8]:
import importlib.metadata
from src.baselines.registry import MODELS as MODEL_SPECS
from src.baselines.runtime import _dgl_compat  # noqa: F401  (import for side effect)
from src.baselines.runtime.dependencies import (
    REQUIREMENT_GROUPS,
    groups_for,
    installed_groups,
)

for group_name, present in sorted(installed_groups().items()):
    backed = [
        spec.name
        for spec in MODEL_SPECS
        if any(group_name in groups_for(spec.name, task) for task in spec.tasks)
    ]
    if present:
        print(f'OK    {group_name:<11} backs {", ".join(backed)}')
    else:
        print(f'MISS  {group_name:<11} disables {", ".join(backed)}')
        missing = ', '.join(REQUIREMENT_GROUPS[group_name].missing_modules())
        print(f'      missing: {missing}')

if installed_groups().get('dgl'):
    import dgl

    from src.baselines.runtime._dgl_compat import dgl_supports_cuda

    dgl_version_info = "unavailable (unknown)"
    try:
        dgl_version_info = dgl.__version__
    except AttributeError:
        try:
            dgl_version_info = importlib.metadata.version("dgl")
        except importlib.metadata.PackageNotFoundError:
            dgl_version_info = "unavailable (metadata not found)"
        except Exception as e:
            dgl_version_info = f"unavailable (error: {type(e).__name__})"

    print()
    print(f'dgl: {dgl_version_info} | CUDA support: {dgl_supports_cuda()}')
    if not dgl_supports_cuda():
        print('  DGL-backed baselines will run on CPU even though torch has a GPU.')

OK    dgl         backs gcn, gat, han, sehgnn, hinormer, pshgcn
OK    h2gb        backs h2gformer
OK    hetero2net  backs hetero2net
OK    sheaf       backs nsd, csnn

dgl: unavailable (metadata not found) | CUDA support: False
  DGL-backed baselines will run on CPU even though torch has a GPU.


In [ ]:
# These five need four different vendor clones. Check them up front rather than
# discovering a missing one after LatGRL has already been tuned.
from src.baselines.registry import get_adapter
from src.baselines.runtime import available_vendors

REQUIRED_VENDORS = ['csnn', 'nsd', 'hinormer', 'h2gb', 'latgrl']
present = available_vendors()
missing = [name for name in REQUIRED_VENDORS if not present.get(name)]
for name in REQUIRED_VENDORS:
    print(('OK  ' if present.get(name) else 'MISSING '), name)
if missing:
    raise RuntimeError(f'Clone these at the repository root first: {missing}')


In [ ]:
import wandb

wandb_key = None
if IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None

if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    wandb.login()


## 1. Campaign settings

In [ ]:
TASK = 'node_classification'

# (model, datasets). HINormer is ACM only; the other four cover all three.
CELLS = [
    ('csnn',       ['acm', 'dblp', 'imdb']),
    ('nsd',        ['acm', 'dblp', 'imdb']),
    ('hinormer',   ['acm']),
    ('h2gformer',  ['acm', 'dblp', 'imdb']),
    ('latgrl',     ['acm', 'dblp', 'imdb']),
]

N_TRIALS = 100
NUM_RUNS = 5
BASE_SEED = 42
EPOCHS = 300

# Units are seeds. To split across machines keep NUM_RUNS and BASE_SEED fixed and
# give each machine a disjoint range; never run the same index concurrently.
UNIT_START = 0
UNIT_COUNT = None

RESUME = False
OUTPUT_DIR = Path('experiments/baselines')
WANDB_PROJECT = 'h2snn_baseline'
WANDB_MODE = 'online'
WANDB_TAGS = ['baselines', TASK]

print(f'{sum(len(datasets) for _, datasets in CELLS)} cells x {NUM_RUNS} seeds '
      f'x {N_TRIALS} trials')


Drop any model whose requirement group did not install. This happens **before** the
launch cells, so a missing library costs those cells and nothing else — and it is
reported, never silent.

In [ ]:
from src.baselines.runtime.dependencies import models_missing_dependencies

requested = [model for model, _ in CELLS]
blocked = models_missing_dependencies(tuple(requested), TASK)
CELLS = [(model, datasets) for model, datasets in CELLS if model not in blocked]

for name, reason in blocked.items():
    print(f'SKIP  {name}: {reason}')
print(f'{len(CELLS)} of {len(requested)} cells runnable: ' + ', '.join(m for m, _ in CELLS))
if not CELLS:
    raise RuntimeError(
        'No runnable models: every requested model is missing a requirement group. '
        'Fix the failures reported by the group-install cell before launching.'
    )

## 2. Confirm the scope

Every selected cell must be runnable and must genuinely need tuning - if any of
these turned out to have published settings, it belongs in the published runner
instead.

In [ ]:
from src.baselines.registry import unsupported_reason

for model, datasets in CELLS:
    adapter = get_adapter(model)
    for dataset in datasets:
        reason = unsupported_reason(model, TASK, dataset)
        assert reason is None, f'{model}/{dataset} is not runnable: {reason}'
        published = adapter.published_params(TASK, dataset)
        space = adapter.search_space(TASK, dataset)
        status = 'PUBLISHED (use run_baseline_published.py)' if published else 'tune'
        print(f'{model:<12}{dataset:<7}{len(space):>3} params  {status}')


## 3. Validate before launching

Resolves every search space and prints the total fit count without running anything.

In [ ]:
import subprocess
import sys


def tuning_command(model, datasets, validate=False):
    cmd = [
        sys.executable, 'scripts/run_baseline_tuning.py',
        '--task', TASK,
        '--models', model,
        '--datasets', *datasets,
        '--output_dir', str(OUTPUT_DIR),
        '--n_trials', str(N_TRIALS),
        '--num_runs', str(NUM_RUNS),
        '--base_seed', str(BASE_SEED),
        '--epochs', str(EPOCHS),
        '--unit_start', str(UNIT_START),
        '--wandb_project', WANDB_PROJECT,
        '--wandb_mode', WANDB_MODE,
        '--wandb_tags', *WANDB_TAGS,
    ]
    if UNIT_COUNT is not None:
        cmd += ['--unit_count', str(UNIT_COUNT)]
    if RESUME:
        cmd.append('--resume')
    if validate:
        cmd.append('--validate_only')
    return cmd


for model, datasets in CELLS:
    subprocess.run(tuning_command(model, datasets, validate=True), check=True)


## 4. Run

One cell per model, so a failure costs one model rather than the batch. Re-run any
of them with `RESUME = True` after a Colab restart: compatible per-seed states
resume their original sweeps, and incompatible ones are preserved while numbered
continuation sweeps are created.

Ordered cheapest first, so a configuration problem surfaces before the expensive
models start. LatGRL is last: it trains an unsupervised encoder and then probes it,
so its wall-clock profile is unlike the other four.

### csnn - ACM, DBLP, IMDB

Legacy in-repo sheaf model. Reads the HGB graph through the shared HomogeneousView plus a per-type input projection.

In [ ]:
model, datasets = 'csnn', ['acm', 'dblp', 'imdb']
cmd = tuning_command(model, datasets)
print(' '.join(cmd))
subprocess.run(cmd, check=True)


### nsd - ACM, DBLP, IMDB

Homogeneous sheaf model, same two adaptations as CSNN. `GeneralSheaf` in the search space scales as d^2 per edge - drop it first if memory bites.

In [ ]:
model, datasets = 'nsd', ['acm', 'dblp', 'imdb']
cmd = tuning_command(model, datasets)
print(' '.join(cmd))
subprocess.run(cmd, check=True)


### hinormer - ACM

ACM only. Uses the authors' own repository, not H2GB's copy, whose transformer stack is commented out.

In [ ]:
model, datasets = 'hinormer', ['acm']
cmd = tuning_command(model, datasets)
print(' '.join(cmd))
subprocess.run(cmd, check=True)


### h2gformer - ACM, DBLP, IMDB

H2GB's GTModel. Rebuilds graphgym's global cfg per trial, so a stale value from a previous trial cannot leak in.

In [ ]:
model, datasets = 'h2gformer', ['acm', 'dblp', 'imdb']
cmd = tuning_command(model, datasets)
print(' '.join(cmd))
subprocess.run(cmd, check=True)


### latgrl - ACM, DBLP, IMDB

Unsupervised encoder, then a probe on the frozen embeddings.

**LatGRL is the one model here that already has published settings** for all
three datasets, transcribed from `LatGRL/utils/params.py`. By the campaign's
usual rule those cells would skip tuning - but the rule assumes the published
values were tuned on *our* graph, and on two of three they were not: their
published ACM and IMDB are the HAN-preprocessed graphs (ACM 4,019 papers /
3 classes; IMDB 4,278 / 3 classes), while ours are HGB's (ACM 3,025; IMDB
4,932 / 5 classes, multi-label). DBLP is the same graph in both.

So: **DBLP could legitimately use the published values**, while ACM and IMDB
are better tuned. The tuning cell below does all three, as requested; the cell
after it is the published-parameter alternative if you would rather take
their DBLP settings as-is. Running both and reporting the better-motivated
one per dataset is also defensible, as long as the table says which.

In [ ]:
model, datasets = 'latgrl', ['acm', 'dblp', 'imdb']
cmd = tuning_command(model, datasets)
print(' '.join(cmd))
subprocess.run(cmd, check=True)


Optional: LatGRL at the authors' published settings instead of tuning. Writes `published_summary.json` beside the tuned `tuning_summary.json`, so running this does not overwrite the cell above.

In [ ]:
published_cmd = [
    sys.executable, 'scripts/run_baseline_published.py',
    '--task', TASK,
    '--models', 'latgrl',
    '--datasets', 'dblp',   # add 'acm', 'imdb' only if you accept the
                            # dataset mismatch described above
    '--output_dir', str(OUTPUT_DIR),
    '--num_runs', str(NUM_RUNS),
    '--base_seed', str(BASE_SEED),
]
print(' '.join(published_cmd))
subprocess.run(published_cmd, check=True)


## 5. Aggregate

Collects whatever has finished into one CSV. Cells that have not run are absent
rather than filled in, so a gap stays visibly a gap.

In [ ]:
import json

import pandas as pd

from src.utils.run_reports import report_metrics_for_task

metrics = report_metrics_for_task(TASK)
wanted = {f'{m}_{TASK}_{d}' for m, ds in CELLS for d in ds}

records = []
for summary_path in sorted(OUTPUT_DIR.glob('*/tuning_summary.json')):
    if summary_path.parent.name not in wanted:
        continue
    summary = json.loads(summary_path.read_text(encoding='utf-8'))['summary']
    model, _, dataset = summary_path.parent.name.partition(f'_{TASK}_')
    row = {'model': model, 'dataset': dataset, 'seeds': summary.get('num_units', 0)}
    for metric in metrics:
        entry = summary.get('metrics', {}).get(metric, {})
        row[metric] = entry.get('mean')
        row[f'{metric}_std'] = entry.get('std')
    records.append(row)

frame = pd.DataFrame(records)
results_path = Path('results') / f'baselines_{TASK}_subset.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
frame.to_csv(results_path, index=False)
print(f'{len(frame)} of {len(wanted)} cells finished -> {results_path}')
frame


In [ ]:
# Manuscript form: mean +/- std on the headline metric.
if not frame.empty:
    headline = metrics[0]
    shown = frame.assign(
        value=frame.apply(
            lambda r: f"{100 * r[headline]:.2f} +/- {100 * r[f'{headline}_std']:.2f}",
            axis=1,
        )
    )
    print(shown.pivot_table(index='model', columns='dataset', values='value',
                            aggfunc='first').to_string())
else:
    print('Nothing finished yet - run the cells in section 4 first.')


## 6. Resume and partition

Per-seed state lives under `OUTPUT_DIR/<cell>/tuning_state/`. Set `RESUME = True`
in section 1 and re-run the relevant cell in section 4 to continue.

In [ ]:
state_root = OUTPUT_DIR / 'tuning_state'
for state_path in sorted(state_root.glob('*/tuning_state_*.json')):
    if state_path.parent.name not in wanted:
        continue
    state = json.loads(state_path.read_text(encoding='utf-8'))
    print(
        f"{state_path.parent.name:<40}{state_path.stem:<24}"
        f"{len(state.get('completed_records', []))}/{state.get('target_trials')} "
        f"sweep {state.get('sweep_id')}"
    )
